In [17]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

# The codes

In [18]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def riskcalc(a,R,p,alfa,r_f):
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        x = x - np.min(x)
    N = len(p)
    risk = h_3(p[rank[0]],alfa)*x[rank[0]]
    for i in range(2,N+1):
        z1 = sum(p[rank[0:i]])
        z2 = sum(p[rank[0:i-1]])
        risk = risk + (h_3(z1,alfa)-h_3(z2,alfa))*x[rank[i-1]]
    risk = risk + extra - (1-sum(a))*r_f
    print("the nominal risk of a:", risk)

In [1]:
def makeset (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    added = []
    for i in range(N):
        new = B[0:i+1]
        N_sets = len(A[i])
        for k in range(N_sets):
            if len(np.intersect1d(A[i][k],new))==len(new):
                break
            if k == N_sets-1:
                A[i].append(new)
                added.append(new)
    return(A,added)

def countsets(sets):
    m = len(sets)
    count = 0
    for k in range(m):
        count = count + len(sets[k])
    return(count)

def convertlist(sets):
    Output = []
    for temp in sets:
        for elem in temp:
            Output.append(elem)
    return(Output)

def solvenominal (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        constraints.append((-R @ a)[i] - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    constraints.append(cp.abs(a)<=1)
    constraints.append(alpha + beta + gamma * (r-1) - (1-cp.sum(a))*r_f + z4 + z2 <= c)
    
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,a.value,v.value,lbda.value,alpha.value,beta.value,gamma.value)
    
    
def robustcheck(a,R,r,c,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        c = c - np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    print("the threshold c:", c+extra)
    print("the robust risk constraint value of a:",prob.value - (1-np.sum(a))*r_f+extra)
    print("is it robust?",(prob.value - (1-np.sum(a))*r_f) <= c)
    #return(prob.value,(prob.value - (1-np.sum(a))*r_f) <= c)
    
                
    

# Results

In [20]:
#np.random.seed(10)
N=4
#x=np.arange(1,N)
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]

In [21]:
p = np.random.rand(N)
p = p/sum(p)
I = 5
R = np.random.rand(N,I)*3-1
sets =psets
print("the probability vector:", p)
print("the states of the risky assets:", R)
print("the sets used:", sets)
print("expected return of the risky assets:",R.transpose().dot(p))

the probability vector: [0.34829293 0.39777368 0.10183611 0.15209728]
the states of the risky assets: [[-0.77607364  0.30763315  1.52168927  0.88144269  0.99068101]
 [-0.86198361 -0.19724905 -0.98823632  0.66726398  1.09917993]
 [ 0.86100827 -0.03157267 -0.50434876  0.72915949 -0.13806794]
 [-0.13096236  1.8780845  -0.03777539  0.41150038  0.1831561 ]]
the sets used: [[0], [1], [2], [3], [0, 1], [0, 2], [0, 3], [1, 2], [1, 3], [2, 3], [0, 1, 2], [0, 1, 3], [0, 2, 3], [1, 2, 3], [0, 1, 2, 3]]
expected return of the risky assets: [-0.54541264  0.31112227  0.07979277  0.70926316  0.79606928]


In [22]:
r = 0.01
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.01
c = 1
print("note: |a|_k<=1 for each k here")
[prob_v,a_value,v_v,lbda_v,alpha_v,beta_v,gamma_v]=solvenominal (sets,p,R,r,m,r_f,c)
print("the optimal expected return of the entire portfolio:", prob_v)
print("the optimal portfolio on the risky assets:", a_value)
print("the optimal portfolio return on the riskfree asset:",(1-sum(a_value))*r_f)
print("the optimal expected return on the risky assets:", prob_v-(1-sum(a_value)*r_f))
robustcheck(a_value,R,r,c,p,m,r_f)
riskcalc(a_value,R,p,m,r_f)


note: |a|_k<=1 for each k here
the optimal expected return of the entire portfolio: 2.4216601261782347
the optimal portfolio on the risky assets: [-1.  1.  1.  1.  1.]
the optimal portfolio return on the riskfree asset: -0.019999999966496308
the optimal expected return on the risky assets: 1.451660126144731
the threshold c: 1.0
the robust risk constraint value of a: 0.8258381341053012
is it robust? True
the nominal risk of a: 0.825838135223759


In [23]:
r = 0.01
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 10
print("note: |a|_k<=1 for each k here")
[prob_v,a_value,v_v,lbda_v,alpha_v,beta_v,gamma_v]=solvenominal (sets,p,R,r,m,r_f,c)
print("the optimal expected return:", prob_v)
print("the optimal portfolio on the risky assets:", a_value)
print("the optimal portfolio return on the riskfree asset:",(1-sum(a_value))*r_f)
print("the optimal expected return on the risky assets:", prob_v-(1-sum(a_value)*r_f))
robustcheck(a_value,R,r,c,p,m,r_f)
riskcalc(a_value,R,p,m,r_f)


note: |a|_k<=1 for each k here
the optimal expected return: 2.439660126117125
the optimal portfolio on the risky assets: [-1.  1.  1.  1.  1.]
the optimal portfolio return on the riskfree asset: -0.001999999997771006
the optimal expected return on the risky assets: 1.4426601261148961
the threshold c: 10.0
the robust risk constraint value of a: 0.8078381355431663
is it robust? True
the nominal risk of a: 0.8078381366616242


In [17]:
r = 0.1
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 1
print("note: |a|_k<=1 for each k here")
[prob_v,a_value,v_v,lbda_v,alpha_v,beta_v,gamma_v]=solvenominal (sets,p,R,r,m,r_f,c)
print("the optimal expected return:", prob_v)
print("the optimal portfolio on the risky assets:", a_value)
print("the optimal portfolio return on the riskfree asset:",(1-sum(a_value))*r_f)
print("the optimal expected return on the risky assets:", prob_v-(1-sum(a_value)*r_f))
robustcheck(a_value,R,r,c,p,m,r_f)
riskcalc(a_value,R,p,m,r_f)

note: |a|_k<=1 for each k here
the optimal expected return: 2.8201146175826883
the optimal portfolio on the risky assets: [1. 1. 1. 1. 1.]
the optimal portfolio return on the riskfree asset: -0.003999999999470403
the optimal expected return on the risky assets: 1.8251146175821589
the threshold c: 1.0
the robust risk constraint value of a: -1.760868112360539
is it robust? True
the nominal risk of a: -1.7608681123720131


In [24]:
r = 0.1
m = 0.1    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 1
print("note: |a|_k<=1 for each k here")
[prob_v,a_value,v_v,lbda_v,alpha_v,beta_v,gamma_v]=solvenominal (sets,p,R,r,m,r_f,c)
print("the optimal expected return:", prob_v)
print("the optimal portfolio on the risky assets:", a_value)
print("the optimal portfolio return on the riskfree asset:",(1-sum(a_value))*r_f)
print("the optimal expected return on the risky assets:", prob_v-(1-sum(a_value)*r_f))
robustcheck(a_value,R,r,c,p,m,r_f)
riskcalc(a_value,R,p,m,r_f)

note: |a|_k<=1 for each k here
the optimal expected return: 2.4396601263914013
the optimal portfolio on the risky assets: [-1.  1.  1.  1.  1.]
the optimal portfolio return on the riskfree asset: -0.001999999999990285
the optimal expected return on the risky assets: 1.4426601263913916
the threshold c: 1.0
the robust risk constraint value of a: -1.35926695986418
is it robust? True
the nominal risk of a: -2.213453500214375


array([0.13022571, 0.71095428, 0.65983816, 0.73290589, 0.32324283])

In [81]:
#print("the v multipliers:", v_v)
#print("the lambda multipliers:", lbda_v)
#print("the alpha multiplier:", alpha_v)
#print("the beta multiplier:", beta_v)
#print("the gamma multiplier:", gamma_v)